# 🚀 Demostración del Orquestador Jerárquico Multi-Agente con LangGraph

Este Jupyter Notebook forma parte del proyecto **Multi-Agent Orchestrator** para Coderhouse.

## 🎯 Objetivos de la Demostración
1. **Demostrar la Arquitectura Jerárquica**: Ver cómo el nodo **Supervisor** centraliza las decisiones de enrutamiento hacia el **Investigador** y el **Analista**.
2. **Prevención del 'Supervisor Infinito'**: Validación de condiciones de suficiencia de datos (`research_data`, `analysis_data`) y control estricto de iteraciones máximas (`step_count >= 5`).
3. **Resiliencia & Ejecución Offline**: Demostrar la compatibilidad con proveedores externos (OpenAI) e integración transparente con `MockLLM` en entornos restringidos sin créditos activa.

In [ ]:
# 1. Instalar dependencias si se ejecuta en Google Colab o entorno limpio
# !pip install -q langgraph langchain-openai langchain-core python-dotenv

## 🛠️ Configuración de Estado y Mocking para LLM
Definimos el esquema del estado global (`TeamState`) utilizando anotaciones de tipo y `add_messages` para la persistencia del historial. Además, implementamos `MockLLM` para garantizar ejecuciones determinísticas sin depender de la cuota de la API.

In [ ]:
import os
from typing import TypedDict, Annotated, Optional, List
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.outputs import ChatResult, ChatGeneration

# --- 1. Definición del Estado del Grafo ---
class TeamState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next_agent: str
    research_data: Optional[str]
    analysis_data: Optional[str]
    step_count: int

# --- 2. Fallback MockLLM para Ejecución Garantizada ---
class MockLLM(BaseChatModel):
    def _generate(self, messages: list[BaseMessage], stop=None, run_manager=None, **kwargs) -> ChatResult:
        last_text = messages[-1].content if messages else ""
        if "investiga" in last_text.lower() or "mercado" in last_text.lower():
            text = "Análisis de mercado: Adopción masiva de orquestadores jerárquicos ReAct estimada en +84% para 2026."
        elif "analiza" in last_text.lower() or "métricas" in last_text.lower():
            text = "Proyección cuantitativa: Incremento de eficiencia operativa del +42.5% con margen de error <2%."
        else:
            text = "Respuesta procesada correctamente por el orquestador."
            
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=text))])

    @property
    def _llm_type(self) -> str:
        return "mock_demo"

## 🧠 Definición de los Nodos del Grafo

### Previene:
- **Supervisor Infinito**: `step_count >= 5` interrumpe forzosamente la ejecución.
- **Contaminación de Contexto**: Los especialistas consumen atributos del estado específicos en lugar de reler todo el historial.

In [ ]:
# --- 3. Nodo Supervisor ---
def supervisor_node(state: TeamState) -> dict:
    step_count = state.get("step_count", 0)
    
    # Salvaguarda 1: Límite estricto de iteraciones
    if step_count >= 5:
        return {"next_agent": "FINISH"}
    
    # Salvaguarda 2: Verificación de suficiencia de datos
    has_research = bool(state.get("research_data"))
    has_analysis = bool(state.get("analysis_data"))
    
    if not has_research:
        next_step = "Investigador"
    elif not has_analysis:
        next_step = "Analista"
    else:
        next_step = "FINISH"
        
    return {"next_agent": next_step}

# --- 4. Nodo Investigador ---
def research_node(state: TeamState) -> dict:
    llm = MockLLM()
    user_query = state["messages"][0].content
    
    # Aislamiento de contexto: solo procesa la consulta necesaria
    prompt = f"Investiga el mercado para: {user_query}"
    res = llm.invoke([HumanMessage(content=prompt)])
    
    return {
        "messages": [HumanMessage(content=f"[INVESTIGADOR]: {res.content}", name="Investigador")],
        "research_data": res.content,
        "step_count": state.get("step_count", 0) + 1
    }

# --- 5. Nodo Analista ---
def analyst_node(state: TeamState) -> dict:
    llm = MockLLM()
    data = state.get("research_data", "")
    
    # Aislamiento de contexto: lee el artefacto de investigación previa
    prompt = f"Analiza métricas basadas en: {data}"
    res = llm.invoke([HumanMessage(content=prompt)])
    
    return {
        "messages": [HumanMessage(content=f"[ANALISTA]: {res.content}", name="Analista")],
        "analysis_data": res.content,
        "step_count": state.get("step_count", 0) + 1
    }

## 🕸️ Construcción y Compilación del Grafo (`StateGraph`)

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(TeamState)

# Registrar nodos
builder.add_node("Supervisor", supervisor_node)
builder.add_node("Investigador", research_node)
builder.add_node("Analista", analyst_node)

# Definir punto de entrada
builder.add_edge(START, "Supervisor")

# Definir mapeo condicional jerárquico
builder.add_conditional_edges(
    "Supervisor",
    lambda state: state["next_agent"],
    {
        "Investigador": "Investigador",
        "Analista": "Analista",
        "FINISH": END
    }
)

# Conectar la devolución de control al Supervisor
builder.add_edge("Investigador", "Supervisor")
builder.add_edge("Analista", "Supervisor")

app = builder.compile()

## 🎬 Ejecución y Streaming de Transiciones en Vivo
Visualizamos en directo las transiciones de estado a través del método `.stream()`.

In [ ]:
initial_state = {
    "messages": [HumanMessage(content="Investigar la adopción de LangGraph y calcular métricas de impacto.")],
    "next_agent": "Supervisor",
    "research_data": None,
    "analysis_data": None,
    "step_count": 0
}

print("=" * 80)
print("🚀 EJECUTANDO ORQUESTADOR JERÁRQUICO MULTI-AGENTE")
print("=" * 80)

for step_output in app.stream(initial_state):
    for node_name, state_update in step_output.items():
        print(f"\n📍 [NODO ALCANZADO]: {node_name}")
        if "next_agent" in state_update:
            print(f"   ➡️ Transición decidida: {state_update['next_agent']}")
        if "research_data" in state_update:
            print(f"   🔎 Resultado Investigación: {state_update['research_data']}")
        if "analysis_data" in state_update:
            print(f"   📊 Resultado Análisis: {state_update['analysis_data']}")
        if "step_count" in state_update:
            print(f"   🔢 Contador de pasos: {state_update['step_count']}")

print("\n" + "=" * 80)
print("✅ FLUJO FINALIZADO CORRECTAMENTE SIN BUCLES INFINITOS")
print("=" * 80)